# Validation Étape 1 — Signal d'apprentissage sur 50 photos UV-A 365 nm

**Objectif** : déterminer en quelques minutes si tes 50 photos contiennent un signal exploitable pour entraîner un modèle dédié, ou s'il faut plus de données / autre approche.

**Pipeline** :
1. Upload des 50 photos
2. Pré-annotation automatique par Claude Opus 4.7
3. Revue rapide humaine
4. Extraction d'embeddings CLIP (vecteurs sémantiques 768-D par image)
5. Visualisation UMAP (projection 2D)
6. Linear probe avec cross-validation 5-fold
7. Verdict + recommandations pour Étape 2

**Durée** : 10-20 min selon la vitesse de ton API + GPU Colab. **Coût** : ~0,50-1 € de tokens Claude pour annoter 50 images. GPU Colab gratuit suffit.

## 1. Installation des dépendances

In [ ]:
!pip install -q anthropic transformers torch torchvision scikit-learn umap-learn matplotlib seaborn pillow
print('OK')

## 2. Imports & configuration

In [ ]:
import os, json, base64, time
from pathlib import Path
import numpy as np
import torch
from transformers import CLIPModel, CLIPProcessor
import anthropic
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

CATEGORIES = ['organic', 'fatty', 'chemical', 'mineral', 'biofilm', 'dust', 'mixed', 'clean']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cpu':
    print('Pas de GPU. Active : Runtime -> Change runtime type -> T4 GPU (gratuit)')

## 3. Upload des 50 photos

Sélectionne tes 50 photos dans la boîte de dialogue qui apparaît.

In [ ]:
from google.colab import files
uploaded = files.upload()
image_dir = Path('/content/uv_images')
image_dir.mkdir(exist_ok=True)
for fname, data in uploaded.items():
    (image_dir / fname).write_bytes(data)
image_paths = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')])
print(f'{len(image_paths)} images chargees dans {image_dir}')

## 4. Clé API Anthropic

Récupère ta clé sur https://console.anthropic.com/settings/keys

In [ ]:
import getpass
ANTHROPIC_API_KEY = getpass.getpass('Anthropic API Key (input invisible) : ')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print('Client API initialise.')

## 5. Pré-annotation automatique avec Claude

Pour chaque photo, Claude classifie le contaminant DOMINANT en une catégorie + intensité + confiance. ~15-30 sec par image.

In [ ]:
LABEL_PROMPT = '''Tu es expert en imagerie UV-A 365 nm pour hygiene industrielle.

Classifie le contaminant DOMINANT visible dans cette photo en repondant UNIQUEMENT en JSON :

{
  "label": "organic" | "fatty" | "chemical" | "mineral" | "biofilm" | "dust" | "mixed" | "clean",
  "intensity": "faible" | "moyenne" | "forte",
  "confidence": 0.0-1.0,
  "rationale": "1 phrase courte decrivant la couleur, texture et localisation"
}

Definitions :
- organic : vert/jaune-vert sature (proteines, sucres, biologique)
- fatty : orange/ambre (lipides, huiles)
- chemical : cyan electrique sature localise (azurants detergent)
- mineral : blanc-bleute granulaire (calcaire, sels)
- biofilm : violet/rose film continu adherent
- dust : tres pale, voile uniforme
- mixed : plusieurs signatures dans la meme zone
- clean : pas de contamination, juste voile UV ambient

Le voile bleu-violet diffus uniforme N EST PAS chemical, c est juste la reflexion UV normale.
Si la surface est propre (juste voile UV), reponds clean.'''

def label_image(img_path):
    with open(img_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    media_type = 'image/jpeg' if img_path.suffix.lower() in ('.jpg', '.jpeg') else 'image/png'
    msg = client.messages.create(
        model='claude-opus-4-7',
        max_tokens=400,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': media_type, 'data': b64}},
                {'type': 'text', 'text': LABEL_PROMPT}
            ]
        }]
    )
    text = msg.content[0].text
    start = text.find('{')
    end = text.rfind('}') + 1
    return json.loads(text[start:end])

labels = {}
for i, img_path in enumerate(image_paths):
    print(f'[{i+1}/{len(image_paths)}] {img_path.name}...', end=' ')
    try:
        labels[img_path.name] = label_image(img_path)
        L = labels[img_path.name]
        print(f"-> {L['label']} ({L.get('intensity', '?')}, conf {L.get('confidence', '?')})")
    except Exception as e:
        labels[img_path.name] = {'label': 'error', 'error': str(e)}
        print(f'ERREUR: {e}')
    time.sleep(0.3)

with open('/content/labels_claude.json', 'w') as f:
    json.dump(labels, f, indent=2, ensure_ascii=False)
print(f'\n{len(labels)} annotations sauvegardees dans /content/labels_claude.json')
from collections import Counter
dist = Counter(l['label'] for l in labels.values() if l.get('label') != 'error')
print('Distribution des labels :', dict(dist))

## 6. Revue visuelle des annotations

Affichage des images avec leur label Claude. Si tu vois des erreurs, télécharge `labels_claude.json`, corrige, ré-uploade avant de continuer.

In [ ]:
n_per_row = 4
n_rows = (len(image_paths) + n_per_row - 1) // n_per_row
fig, axes = plt.subplots(n_rows, n_per_row, figsize=(16, 4 * n_rows))
axes = axes.flatten() if n_rows > 1 else ([axes] if n_per_row == 1 else axes)
for i, img_path in enumerate(image_paths):
    ax = axes[i]
    img = Image.open(img_path)
    ax.imshow(img)
    label_data = labels.get(img_path.name, {})
    label = label_data.get('label', '?')
    intensity = label_data.get('intensity', '')
    conf = label_data.get('confidence', 0)
    title = f"{i+1}. {label}\n{intensity} . {conf}"
    ax.set_title(title, fontsize=9)
    ax.axis('off')
for j in range(len(image_paths), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

## 7. Extraction des embeddings CLIP

Chaque image -> vecteur 768D résumant son contenu sémantique. Modèle : `clip-vit-large-patch14`.

In [ ]:
print('Telechargement de CLIP-ViT-Large-patch14 (env. 1.7 Go)...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
clip_model.train(False)  # mode inference (equivalent .eval() en PyTorch)
clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
print('OK. Extraction des embeddings...')

embeddings = []
filenames = []
for img_path in image_paths:
    pil = Image.open(img_path).convert('RGB')
    inputs = clip_proc(images=pil, return_tensors='pt').to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    embeddings.append(emb.cpu().numpy()[0])
    filenames.append(img_path.name)
embeddings = np.stack(embeddings)
print(f'Shape : {embeddings.shape} (devrait etre ({len(image_paths)}, 768))')

## 8. Linear probe — Régression logistique sur embeddings, cross-validation 5-fold

**Interprétation** :
- **Accuracy > 70 %** : signal très fort, passer à Étape 2 (segmentation pixel)
- **Accuracy 50-70 %** : signal présent, fine-tuning CLIP recommandé
- **Accuracy < 50 %** : signal faible, plus de données ou modèle plus puissant

Random baseline pour 7 classes ≈ 14 %.

In [ ]:
y_all = np.array([labels[fn].get('label', 'error') for fn in filenames])
valid_mask = (y_all != 'error') & (y_all != '?')
X = embeddings[valid_mask]
y = y_all[valid_mask]
print(f'Echantillons valides : {len(y)}')
print(f'Classes uniques : {sorted(set(y))}')

from collections import Counter
class_counts = Counter(y)
print('\nDistribution :')
for cat, n in class_counts.most_common():
    print(f'  {cat:12s} {n:3d} echantillons')

viable_classes = {c for c, n in class_counts.items() if n >= 2}
if len(viable_classes) < len(class_counts):
    drop = set(class_counts) - viable_classes
    print(f'\nClasses ignorees (< 2 echantillons) : {drop}')
    keep = np.array([yi in viable_classes for yi in y])
    X, y = X[keep], y[keep]

if len(set(y)) < 2:
    print('\nPas assez de classes pour entrainer.')
else:
    n_splits = min(5, min(Counter(y).values()))
    print(f'\nCross-validation {n_splits}-fold...')
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    preds = np.empty_like(y, dtype=object)
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')
        clf.fit(X[tr], y[tr])
        preds[te] = clf.predict(X[te])

    print('\n=== RESULTAT CROSS-VALIDE ===\n')
    print(classification_report(y, preds, zero_division=0))

    cm_labels = sorted(set(y))
    cm = confusion_matrix(y, preds, labels=cm_labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cm_labels, yticklabels=cm_labels)
    plt.xlabel('Predit')
    plt.ylabel('Verite (Claude)')
    plt.title('Matrice de confusion -- Linear probe sur embeddings CLIP')
    plt.tight_layout()
    plt.show()

    accuracy = (preds == y).mean()
    print(f'\nAccuracy globale : {accuracy*100:.1f} %')
    if accuracy > 0.70:
        print('   SIGNAL TRES FORT -- passe a Etape 2 (segmentation pixel U-Net)')
    elif accuracy > 0.50:
        print('   SIGNAL PRESENT -- fine-tuning CLIP recommande sur 200-500 images')
    else:
        print('   SIGNAL FAIBLE -- soit augmenter le dataset (>200 images), soit revoir les annotations')

## 9. Visualisation UMAP

Projection 2D de l'espace 768D. Si les couleurs (catégories) forment des clusters distincts, le signal est apprenable.

In [ ]:
n_samples = len(X)
n_neighbors = min(15, max(2, n_samples - 1))
reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, random_state=42, metric='cosine')
X_2d = reducer.fit_transform(X)

plt.figure(figsize=(11, 8))
palette = sns.color_palette('Set2', n_colors=len(set(y)))
for cat, color in zip(sorted(set(y)), palette):
    mask = y == cat
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=f'{cat} (n={mask.sum()})', s=120, alpha=0.75, edgecolors='black', linewidth=0.5, color=color)
plt.legend(loc='best', framealpha=0.9)
plt.title('Projection UMAP des embeddings CLIP')
plt.xlabel('UMAP-1')
plt.ylabel('UMAP-2')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print('Lecture :')
print(' - Couleurs en nuages compacts distincts -> signal fort')
print(' - Couleurs melangees partout -> CLIP zero-shot insuffisant')
print(' - Couleur eclatee en sous-clusters -> variabilite intra-classe importante')

## 10. Téléchargement des résultats

In [ ]:
np.savez('/content/clip_embeddings.npz', embeddings=embeddings, filenames=np.array(filenames))
print('Fichiers generes :')
print('  /content/labels_claude.json    -- annotations Claude')
print('  /content/clip_embeddings.npz   -- embeddings 768-D + noms de fichiers')
from google.colab import files as gfiles
gfiles.download('/content/labels_claude.json')
gfiles.download('/content/clip_embeddings.npz')

## 11. Verdict & prochaines étapes

### Cas A — Accuracy > 70 %
Tes 50 photos contiennent un signal exploitable. **Passe à Étape 2** :
1. Annoter au pixel (masques de segmentation) sur 50-100 images via CVAT/LabelStudio
2. Entraîner un U-Net
3. Combiner U-Net (où) + Claude (quoi)

### Cas B — Accuracy 50-70 %
Signal limité. Trois options :
1. Doubler/tripler le dataset (200-500 images)
2. Fine-tuner CLIP (vrai entraînement) — gain typique +15-20 %
3. Tester DINOv2 ou SigLIP (encodeurs plus puissants)

### Cas C — Accuracy < 50 %
1. Vérifier les annotations Claude (revue manuelle, section 6)
2. Constater que les classes sont trop ambiguës dans tes images
3. Investir dans annotations professionnelles + dataset plus grand

---

**Envoie-moi les résultats de la section 8** (accuracy + matrice confusion + UMAP) et je te dis quel chemin emprunter.